In [32]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/nppe-dlp-2026-term-1/sample_submission.csv
/kaggle/input/competitions/nppe-dlp-2026-term-1/train.csv
/kaggle/input/competitions/nppe-dlp-2026-term-1/test.csv


# Install Required Libraries

In [33]:
!pip install -q transformers datasets peft accelerate bitsandbytes trl

In [34]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Import Libraries

In [35]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

# Load Dataset

In [36]:
train_df = pd.read_csv("/kaggle/input/competitions/nppe-dlp-2026-term-1/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/nppe-dlp-2026-term-1/test.csv")

train_df.head()

,ID,sentence,label,language
0,82,ਇਹ ਫਿਲਮ ਇੱਕ ਬੇਹਤਰੀਨ ਕਹਾਣੀ ਸੁਣਾਉਣ ਦਾ ਸਭ ਤੋਂ ਵਧੀ...,Negative,pa
1,618,"ਇੱਕ ਸਮਗਰੀ ਦੇ ਰੂਪ ਵਿੱਚ, ਕੋਟਿੰਗ ਤੋਂ ਬਿਨਾਂ, ਸਿਰਫ ...",Negative,pa
2,812,"ബ്രിസിലുകൾ കട്ടിയുള്ള പ്ലാസ്റ്റിക് ആണ്, അതിനാൽ...",Negative,ml
3,304,এটি বিআইএস প্রত্যয়িত এবং নিরাপদ বলে দাবি করা হয়।,Positive,bn
4,295,6 mAh બેટરી અને લાંબા સમય સુધી ચાલતી નથી.,Negative,gu


## Check distribution
If the dataset is balanced or not

In [37]:
train_df['label'].value_counts()

label
Positive    456
Negative    444
Name: count, dtype: int64

# Replace Language Codes With Full 
LLMs understand natural language names much better.

In [38]:
lang_map = {
"as":"Assamese",
"bd":"Bodo",
"bn":"Bengali",
"gu":"Gujarati",
"hi":"Hindi",
"kn":"Kannada",
"ml":"Malayalam",
"mr":"Marathi",
"or":"Odia",
"pa":"Punjabi",
"ta":"Tamil",
"te":"Telugu",
"ur":"Urdu"
}

In [39]:
train_df["language_name"] = train_df["language"].map(lang_map)
test_df["language_name"] = test_df["language"].map(lang_map)

In [40]:
train_df.head()

,ID,sentence,label,language,language_name
0,82,ਇਹ ਫਿਲਮ ਇੱਕ ਬੇਹਤਰੀਨ ਕਹਾਣੀ ਸੁਣਾਉਣ ਦਾ ਸਭ ਤੋਂ ਵਧੀ...,Negative,pa,Punjabi
1,618,"ਇੱਕ ਸਮਗਰੀ ਦੇ ਰੂਪ ਵਿੱਚ, ਕੋਟਿੰਗ ਤੋਂ ਬਿਨਾਂ, ਸਿਰਫ ...",Negative,pa,Punjabi
2,812,"ബ്രിസിലുകൾ കട്ടിയുള്ള പ്ലാസ്റ്റിക് ആണ്, അതിനാൽ...",Negative,ml,Malayalam
3,304,এটি বিআইএস প্রত্যয়িত এবং নিরাপদ বলে দাবি করা হয়।,Positive,bn,Bengali
4,295,6 mAh બેટરી અને લાંબા સમય સુધી ચાલતી નથી.,Negative,gu,Gujarati


# Create Prompt Format

Gemma is instruction-tuned, convert samples into instructions.


In [41]:
def format_prompt(row):
    
    prompt = f"""You are a sentiment analysis assistant.

Task: Determine whether the sentiment of the sentence is Positive or Negative.

Language: {row['language_name']}
Sentence: {row['sentence']}

Respond with only one word: Positive or Negative.

Answer: {row['label']}"""

    return prompt

train_df["text"] = train_df.apply(format_prompt, axis=1)

## Convert to HuggingFace dataset:

In [42]:
dataset = Dataset.from_pandas(train_df[["text"]])

In [ ]:
from huggingface_hub import login

login(token="HF_TOKEN") # add your HF Token

In [44]:
# Verify GPU in Code
print("CUDA Available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA Available: True
GPU: Tesla T4


In [45]:
# Set Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


# Load Gemma Model (4-bit QLoRA)

In [46]:
model_name = "google/gemma-3-1b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

## Load tokenizer:

In [47]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

## Load model:

In [48]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    # device_map="auto"
    device_map={"":0} 
)

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

# Apply LoRA

In [49]:
lora_config = LoraConfig(
    r=24,
    lora_alpha=48,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

## Check trainable params:

In [50]:
model.print_trainable_parameters()

trainable params: 4,472,832 || all params: 1,004,358,784 || trainable%: 0.4453


In [51]:
# torch.backends.cuda.matmul.allow_tf32 = True

# Training Configuration

1 — Create SFT Config\
2 — Initialize Trainer

In [52]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir="./results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=20,
    # fp16=True,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=sft_config,
    processing_class=tokenizer
)


Adding EOS to train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

In [53]:
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0))

1
Tesla T4


# Train

In [54]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'pad_token_id': 1}.


Step,Training Loss
20,3.083696
40,2.304819
60,2.293245
80,2.298347
100,2.231381
120,2.181565
140,2.074109
160,2.001855
180,1.959462
200,2.026318


TrainOutput(global_step=339, training_loss=2.1094845346996567, metrics={'train_runtime': 1176.8386, 'train_samples_per_second': 2.294, 'train_steps_per_second': 0.288, 'total_flos': 1290771414935040.0, 'train_loss': 2.1094845346996567})

# Create Inference Prompt

In [63]:
def create_test_prompt(row):

    prompt = f"""You are a sentiment analysis assistant.

Task: Determine whether the sentiment of the sentence is Positive or Negative.

Language: {row['language_name']}
Sentence: {row['sentence']}

Respond with only one word: Positive or Negative.

Answer:"""

    return prompt

# Generate Predictions

In [64]:
def predict(text):
    
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=5,
        temperature=0.1
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return response

# def predict_vote(prompt):

#     votes = []

#     for _ in range(5):

#         inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=3,
#             temperature=0.1
#         )

#         decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

#         votes.append(extract_label(decoded))

#     return max(set(votes), key=votes.count)

In [65]:
def extract_label(output):

    output = output.split("Answer:")[-1].strip().lower()

    if output.startswith("positive"):
        return 1
    elif output.startswith("negative"):
        return 0
    else:
        return 0

# def extract_label(output):
#     text = output.split("Answer")[-1].lower()

#     if "negative" in text:
#         return 0
#     if "positive" in text:
#         return 1
#     return 0

# Run predictions:

In [66]:
predictions = []

for _, row in test_df.iterrows():

    prompt = create_test_prompt(row)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=3,
        temperature=0.1
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    label = extract_label(decoded)

    predictions.append(label)

# predictions = []

# for _, row in test_df.iterrows():

#     prompt = create_test_prompt(row)

#     label = predict_vote(prompt)

#     predictions.append(label)

In [67]:
# for i in range(5):

#     row = test_df.iloc[i]

#     prompt = create_test_prompt(row)

#     inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

#     outputs = model.generate(**inputs, max_new_tokens=5)

#     decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

#     print("MODEL OUTPUT:\n", decoded)
#     print()

In [68]:
# predictions = []

# for _, row in test_df.iterrows():
    
#     prompt = create_test_prompt(row)
    
#     output = predict(prompt)

#     if "Positive" in output:
#         predictions.append(1)
#     else:
#         predictions.append(0)

# Create Submission File

In [69]:
submission = pd.DataFrame({
    "ID": test_df["ID"],
    "label": predictions
})

submission.to_csv("submission.csv", index=False)

In [70]:
submission.head()

,ID,label
0,550,1
1,397,1
2,757,1
3,407,0
4,294,1
